# 06 · Сравнение методов

Ноутбук ничего не считает: читает `runs/*.json` и раскладывает рядом. Модель не нужна,
открывается на любой машине.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

runs = report.load_runs()
print("прогонов:", ", ".join(runs) or "пока нет")
extended = {k: v for k, v in runs.items() if k.endswith("-extended")}
product = {k: v for k, v in runs.items() if k.endswith("-product")}

## Сводка

Расширенный тест — основной: 100 ситуаций, интервал около 10 пунктов в каждую сторону.
Тест продукта — правда продукта, но 33 ситуации и 16 пунктов, его читаем как подтверждение,
а не как измерение.

In [ ]:
print("РАСШИРЕННЫЙ ТЕСТ")
print(report.table(extended))
print()
print("ТЕСТ ПРОДУКТА")
print(report.table(product))

In [ ]:
fig = report.bars(extended, title="расширенный тест: усы — 95 % интервал Уилсона")

## Изменение к базовой линии

In [ ]:
base = extended.get("base-extended")
for name, run in extended.items():
    if base and name != "base-extended":
        print("═" * 60, name)
        print(report.deltas(base, run))

## По категориям

Средняя доля выполненных проверок в каждой категории. Здесь видно, где метод помог,
а где просел: рост среднего иногда куплен падением в одной категории.

In [ ]:
print(report.category_table(extended))

## Один запрос у всех методов

In [ ]:
rows = list(data.load("test_extended"))
row = rows[60]
print("ЗАПРОС:", data.request(row))
for name, run in extended.items():
    print("═" * 78, name)
    print(run["answers"].get(row["id"], "нет ответа"))

## Что считать результатом

- разница меньше половины ширины интервала — не результат: на расширенном тесте
  это десять пунктов, на тесте продукта пятнадцать;
- сдвиг на расширенном тесте должен подтверждаться на тесте продукта по знаку;
- рост ложных отказов обесценивает любой рост остальных метрик;
- preference accuracy и perplexity из ноутбуков обучения объясняют, *почему* метод сдвинул
  метрики: SFT снижает perplexity, методы на парах поднимают preference accuracy.